# The Quality of the Question: AI Podcast Studio

**Description:**
A Python application that takes an excerpt from the entrepreneurship themed book, *The Quality of the Question*, and transforms it into a structured podcast episode.

**Repository:**
https://github.com/paolahsp/The-Quality-of-the-Question-AI-Podcast-Studio


**Notice:**
The source manuscript, *The Quality of the Question*, is © 2026 Paola Hintze. All rights reserved. This repository contains only an authorized excerpt for educational demonstration purposes. The original manuscript may not be reproduced or redistributed.

The Quality of the Question: AI Podcast Studio software developed by Paola Hintze, Marja Wiegand, and John Adams.

## Step 0: Pseudocode (Backend Development)

```
1. Setup environment
   - load required packages and libraries
   - load API key, initialize OpenAI client
   - validate loading success

2. Load and validate source text
   - read book excerpt from file
   - validate correct file types
   - validate not empty / not invalid

3. Define the JSON structure
   - Pydantic model for structured output
     (episode_title, opening_hook, book_introduction, book_journey,
     chapter_one_story, core_lesson, key_takeaways, reflection_question,
     listener_action, closing_teaser)
   - build system prompt
   - build user prompt with the source excerpt

4. LLM processing
   - call LLM, request structured JSON output
   - validate response against the Pydantic model

5. Output script for pass/fail review
   - present generated script to the user (frontend handles UI)
   - pass: continue to audio generation
   - fail: re-run step 4 to regenerate the script

6. Generate audio from approved script
   - call TTS API
   - save as MP3
   - validate output file was created / is playable

7. Output files
   - save transcript (txt/json)
   - save audio (mp3)
   - make both available for download
```

## Step 1: Import necessary packages and load instances

In [1]:
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import List
from dotenv import load_dotenv

print("Loading environment...")

# Load API key from .env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "⚠️ The API key was not found in the .env file.\n"
        "How to fix: create a .env file in the repo root and set OPENAI_API_KEY=replace_with_your_key, then restart the kernel. "
    )

# Initialize client
client = OpenAI(api_key=api_key)

print("\n✅ Environment loaded successfully. OpenAI client initialized.")

Loading environment...

✅ Environment loaded successfully. OpenAI client initialized.


## Step 2: Load input from source

In [2]:
SOURCE_PATH = "data/source_texts/input_text_test_pooh.md"
ALLOWED_EXTENSIONS = (".txt", ".md")

print("Loading source text...")

# Validate file type before reading
if not SOURCE_PATH.lower().endswith(ALLOWED_EXTENSIONS):
    raise ValueError(
        f"⚠️ Unsupported file type for '{SOURCE_PATH}'. "
        f"Allowed types: {', '.join(ALLOWED_EXTENSIONS)}\n"
    )

# Read the source file
with open(SOURCE_PATH, "r", encoding="utf-8") as f:
    source_text = f.read().strip()

# Validate it isn't empty
if not source_text:
    raise ValueError(f"⚠️ Source file '{SOURCE_PATH}' is empty. Add content and try again.")

word_count = len(source_text.split())

print(f"✅ Source loaded successfully: {SOURCE_PATH}")
print('\n' + '=' * 50)
print('OUTPUT')
print('=' * 50)
print(f"Word count: {word_count}")
print(f"Preview: {source_text[:200]}...")

Loading source text...
✅ Source loaded successfully: data/source_texts/input_text_test_pooh.md

OUTPUT
Word count: 2170
Preview: Here is Edward Bear, coming downstairs now, bump, bump, bump, on the back of his head, behind Christopher Robin. It is, as far as he knows, the only way of coming downstairs, but sometimes he feels th...


## Step 3: Define the JSON structure

In [3]:
from pydantic import BaseModel

# Structured output schema
class PodcastScript(BaseModel):
    episode_title: str
    opening_hook: str
    book_introduction: str
    book_journey: str
    chapter_one_story: str
    core_lesson: str
    key_takeaways: list[str]
    reflection_question: str
    listener_action: str
    closing_teaser: str

SYSTEM_PROMPT = """You are an expert podcast script writer adapting book excerpts into spoken-word episodes.

Rules you must follow:
- Use only information contained in the provided source excerpt.
- Preserve the author's central argument and meaning.
- Do not invent quotations, statistics, personal experiences, business cases, or biographical information.
- Do not copy the source excerpt word for word.
- Adapt the material into a natural spoken podcast format.
- Use one narrator.
- Maintain a thoughtful, intelligent, direct, and human tone.
- Include an opening hook, key takeaways, one reflection question, and one practical action.
- Return the result using the required structured format.
"""

print("✅ JSON structure and prompts defined")

✅ JSON structure and prompts defined


## Step 4: LLM processing

In [4]:
print("Transforming source text with LLM...")

try:
    completion = client.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": source_text}
        ],
        response_format=PodcastScript,
        # Add in a max token spend for testing, can be removed after validation
        max_tokens=1500,
    )
except Exception as e:
    raise RuntimeError(f"⚠️ LLM request failed: {e}. Check connection and API key, then retry.")

# Validate the response parsed successfully against the Pydantic model
podcast_script = completion.choices[0].message.parsed

if podcast_script is None:
    raise ValueError("⚠️ Model did not return a valid structured response. Try again.")

print("✅ Script generated and validated successfully.")
print('\n' + '=' * 50)
print('OUTPUT')
print('=' * 50)
print(podcast_script.model_dump_json(indent=2))

Transforming source text with LLM...
✅ Script generated and validated successfully.

OUTPUT
{
  "episode_title": "Winnie-the-Pooh: The Quest for Honey",
  "opening_hook": "What do you do when you need something sweet? For Winnie-the-Pooh, it’s a journey filled with creativity, a little bit of mischief, and quite a few bumps, both literal and figurative. Join me as we explore this charming tale of adventure and resourcefulness.",
  "book_introduction": "Today, we're diving into the whimsical world of A.A. Milne's beloved character, Winnie-the-Pooh. Who could forget the lovable bear who's obsessed with honey and always finds himself in hilarious predicaments? In this episode, we'll see just how far Pooh goes in his quest for sweetness, aided by his best friend, Christopher Robin.",
  "book_journey": "In today's story, we find Pooh contemplating the buzzing sound of bees, setting off a series of thoughts that direct him on a journey towards his favorite treat. It's a classic adventure fil

## Step 5: Output script for pass/fail review

In [ ]:
# In the web app, the frontend collects pass/fail from the user.
# This notebook simulates that decision with an input() prompt.
approved = False
RETRY_NOTE = "\n\nNote: a previous attempt at this script was rejected. Try a different angle, opening hook, or structure this time."

while not approved:
    print('\n' + '=' * 50)
    print('SCRIPT FOR REVIEW')
    print('=' * 50)
    print(podcast_script.model_dump_json(indent=2))

    decision = input("\nPass or fail this script? Please type only Pass or Fail, then press enter. ").strip().lower()

    if decision == "pass":
        approved = True
        print("\n✅ Script approved. Continuing to audio generation.")
    elif decision == "fail":
        print("⚠️ Script failed. Regenerating with a different approach...")
        completion = client.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": source_text + RETRY_NOTE}
            ],
            response_format=PodcastScript,
            max_tokens=1500,
        )
        podcast_script = completion.choices[0].message.parsed
    else:
        print("Please select 'pass' or 'fail' to continue.")


SCRIPT FOR REVIEW
{
  "episode_title": "Winnie-the-Pooh: The Quest for Honey",
  "opening_hook": "What do you do when you need something sweet? For Winnie-the-Pooh, it’s a journey filled with creativity, a little bit of mischief, and quite a few bumps, both literal and figurative. Join me as we explore this charming tale of adventure and resourcefulness.",
  "book_introduction": "Today, we're diving into the whimsical world of A.A. Milne's beloved character, Winnie-the-Pooh. Who could forget the lovable bear who's obsessed with honey and always finds himself in hilarious predicaments? In this episode, we'll see just how far Pooh goes in his quest for sweetness, aided by his best friend, Christopher Robin.",
  "book_journey": "In today's story, we find Pooh contemplating the buzzing sound of bees, setting off a series of thoughts that direct him on a journey towards his favorite treat. It's a classic adventure filled with his characteristic innocence and lack of understanding of how th